# Canonical LoRA training (Seeds of Truth)

This notebook is a thin walk-through of the training pipeline in
`seedsoftruth.train_ai`. Every cell calls into the package — the
heavy lifting lives in `train.py`, so anything you change here is
also available from the CLI.

For the full walk-through with no notebook, run:

```
python -m seedsoftruth.train_ai \
    --mode corpus \
    --base-model unsloth/Qwen3-8B \
    --dataset-dir ./corpus \
    --output-dir ./lora_out
```


## 1. Configure the run

Edit the dataclass fields below to match your run. Required fields are `base_model`, `dataset_dir`, and `output_dir`.

In [ ]:
from seedsoftruth.train_ai import TrainingConfig

config = TrainingConfig(
    # ---- required ----
    base_model="unsloth/Qwen3-8B",        # HF repo or local path
    dataset_dir="./corpus",                # .txt files (corpus mode) or .json/.jsonl (qa mode)
    output_dir="./lora_out",

    # ---- mode ----
    mode="corpus",                         # "corpus" or "qa"

    # ---- LoRA ----
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.1,
    lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],

    # ---- training ----
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    eval_split_frac=0.1,                   # 0 disables eval
)

config


## 2. Load the base model

Uses Unsloth if installed, falls back to `transformers` otherwise.

In [ ]:
from seedsoftruth.train_ai import load_model_and_tokenizer, apply_lora

base_model, tokenizer = load_model_and_tokenizer(config)
peft_model = apply_lora(base_model, tokenizer, config)
peft_model


## 3. Load and tokenize the dataset

Corpus mode reads `.txt` files; QA mode reads `{question, answer}` records from `.json` / `.jsonl`. Both paths feed `tokenize_and_pack`, which appends EOS between docs and packs into fixed-size blocks.

In [ ]:
from seedsoftruth.train_ai import (
    load_corpus_dataset, load_qa_dataset, tokenize_and_pack,
)
from datasets import Dataset

if config.mode == "corpus":
    raw_texts = load_corpus_dataset(config.dataset_dir)
else:
    raw_texts = load_qa_dataset(config.dataset_dir)

ds = Dataset.from_list([{"text": t} for t in raw_texts])
split = ds.train_test_split(test_size=config.eval_split_frac, seed=config.seed)         if config.eval_split_frac > 0 else None

if split is not None:
    train_dataset = tokenize_and_pack(list(split["train"]["text"]), tokenizer, config.block_size)
    eval_dataset = tokenize_and_pack(list(split["test"]["text"]), tokenizer, config.block_size)
else:
    train_dataset = tokenize_and_pack(raw_texts, tokenizer, config.block_size)
    eval_dataset = None

print(f"train blocks: {len(train_dataset)}; eval blocks: {len(eval_dataset) if eval_dataset else 0}")


## 4. Train

The pre-built trainer wires up `TrainingArguments`, optional early stopping, and saves the adapter when training finishes.

In [ ]:
from seedsoftruth.train_ai import build_trainer

trainer = build_trainer(peft_model, tokenizer, train_dataset, eval_dataset, config)
trainer.train(resume_from_checkpoint=config.resume_from_checkpoint)
print("Training complete")

trainer.model.save_pretrained(config.output_dir)
tokenizer.save_pretrained(config.output_dir)


## 5. (Optional) Resume from a checkpoint

Pass the path to the `checkpoint-<step>` directory you want to resume from.

In [ ]:
# Run only if you're resuming a previous run.
# CHECKPOINT_DIR = f"{config.output_dir}/checkpoint-500"
# trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)


## 6. (Optional) Push the LoRA adapter to HuggingFace

Set `HF_TOKEN` in the environment first. Defaults to a private repo.

In [ ]:
# from seedsoftruth.train_ai import push_to_hub
#
# push_to_hub(
#     local_dir=config.output_dir,
#     repo_id="your-org/your-lora-name",
#     private=True,
# )


## 7. (Optional) Merge the LoRA adapter into the base model

After training and pushing the adapter, you can fuse it into a single standalone checkpoint with `seedsoftruth.train_ai.merge_lora`.

In [ ]:
# from seedsoftruth.train_ai.merge_lora import merge_lora
#
# merge_lora(
#     base_model=config.base_model,
#     adapter=config.output_dir,
#     output_dir="./merged_out",
#     dtype="bfloat16",
#     push_to_hub=None,        # or "your-org/your-merged-name"
#     sanity_check=True,
# )
